In [1]:
import numpy as np
import scipy as sp
import requests
import os
import gzip
from sklearn.covariance import empirical_covariance, shrunk_covariance
from statsmodels.stats.multivariate import test_cov_oneway, test_cov
import random
import itertools
from typing import NamedTuple
from tqdm.notebook import trange,tqdm
from tqdm.contrib.concurrent import process_map
import cProfile, pstats, io
from pstats import SortKey
from multiprocessing.pool import ThreadPool as Pool

In [2]:
data_sources = {
    "training_images": "train-images-idx3-ubyte.gz",  # 60,000 training images.
    "test_images": "t10k-images-idx3-ubyte.gz",  # 10,000 test images.
    "training_labels": "train-labels-idx1-ubyte.gz",  # 60,000 training labels.
    "test_labels": "t10k-labels-idx1-ubyte.gz",  # 10,000 test labels.
}

In [3]:
data_sources

{'training_images': 'train-images-idx3-ubyte.gz',
 'test_images': 't10k-images-idx3-ubyte.gz',
 'training_labels': 'train-labels-idx1-ubyte.gz',
 'test_labels': 't10k-labels-idx1-ubyte.gz'}

In [4]:


data_dir = "../_data"
os.makedirs(data_dir, exist_ok=True)

base_url = "https://ossci-datasets.s3.amazonaws.com/mnist/"

for fname in data_sources.values():
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print("Downloading file: " + fname)
        resp = requests.get(base_url + fname, stream=True)
        resp.raise_for_status()  # Ensure download was succesful
        with open(fpath, "wb") as fh:
            for chunk in resp.iter_content(chunk_size=128):
                fh.write(chunk)

In [5]:
mnist_dataset = {}

# Images
for key in ("training_images", "test_images"):
    with gzip.open(os.path.join(data_dir, data_sources[key]), "rb") as mnist_file:
        mnist_dataset[key] = np.frombuffer(
            mnist_file.read(), np.uint8, offset=16
        ).reshape(-1, 28 * 28)
# Labels
for key in ("training_labels", "test_labels"):
    with gzip.open(os.path.join(data_dir, data_sources[key]), "rb") as mnist_file:
        mnist_dataset[key] = np.frombuffer(mnist_file.read(), np.uint8, offset=8)

In [6]:
x_train, y_train, x_test, y_test = (
    mnist_dataset["training_images"],
    mnist_dataset["training_labels"],
    mnist_dataset["test_images"],
    mnist_dataset["test_labels"],
)

In [7]:
options = np.unique(y_train)
covariances = []
nobs = []
means = []
for option in options:
    cov = np.cov(x_train[y_train==option],rowvar=False,ddof=1)
    cov_s = shrunk_covariance(cov)
    nobs.append(len(x_train[y_train==option]))
    covariances.append(cov_s)
    means.append(np.mean(x_train[y_train==option],axis=0))
#test_cov_oneway(covariances,nobs)

In [8]:
# Generates the Fisher Linear Discriminant for 0 and 1
w01t = np.linalg.inv(covariances[0]+covariances[1])@(means[0]-means[1])
w01 = w01t/np.linalg.norm(w01t)

In [9]:
# Picks a "good" value to seperate these two catagories
w01 @(means[0]+means[1])/2

np.float64(-138.07207359962857)

In [10]:
# Figure out which side of the seperator is which
w01@means[0]

np.float64(194.91931430780045)

In [11]:
# Picks a random element and checks what the overlap of it is
random.choice(x_test[y_test==0])@w01

np.float64(187.01317057868025)

In [49]:
# Fisher Linear Discriminant Analysis classifier
# This will create a set of cascading vectors, breakpoints
# and scales to determine when we have what type of image
ws = []
cs = []
scales = []
for option in options:
    ins = x_train[y_train ==option]
    outs = x_train[y_train!=option]
    covin = np.cov(ins,rowvar=False,ddof=1)
    covout = np.cov(outs,rowvar=False,ddof=1)
    mean_in = np.mean(ins,axis=0)
    mean_out = np.mean(outs,axis=0)
    wt = np.linalg.pinv(covout+covin,hermitian=True)@(mean_in-mean_out)
    wp = wt/np.linalg.norm(wt)
    ct = wp @(mean_in+mean_out)/2
    cin = wp @ mean_in
    scale = cin-ct
    ws.append(wp)
    cs.append(ct)
    scales.append(scale)


In [13]:
cs[0]

np.float64(45.213284475366564)

In [14]:
scales

[np.float64(250.87690286657448),
 np.float64(260.12434897167685),
 np.float64(177.09481149395577),
 np.float64(189.30495546596865),
 np.float64(157.78771609505642),
 np.float64(136.28145316096655),
 np.float64(228.37411905818936),
 np.float64(223.4892170605805),
 np.float64(166.38346565279636),
 np.float64(158.4251165158999)]

In [15]:
def classify(datapoint,verbose = False):
    bestval = -np.inf
    bestoption = "invalid"
    for i in range(len(options)):
        conf = (ws[i]@datapoint-cs[i])/scales[i]
        if conf > bestval:
            bestoption = options[i]
            bestval = conf
        if verbose:
            print("For {} got value {}".format(options[i],conf))
    return bestoption

In [50]:
classify(x_test[100])

np.uint8(6)

In [17]:
y_test[100]

np.uint8(6)

In [18]:
i = random.randrange(10000)
y_class = classify(x_test[i])
y_truth = y_test[i]
if y_class != y_truth:
    print(i)
else:
    print("Success")

6042


In [51]:
classify(x_test[6847])

np.uint8(4)

In [20]:
y_test[6847]

np.uint8(6)

In [52]:
np.sum([classify(x_train[i]) for i in range(60000)] != y_train)

np.int64(6529)

In [22]:
classify(x_test[30],verbose=True)

For 0 got value -1.3674814376691238
For 1 got value -0.9503484792336436
For 2 got value -2.198338094395112
For 3 got value 1.768556288114883
For 4 got value -1.7702450582835096
For 5 got value 0.13138269260972052
For 6 got value -0.6675132613498416
For 7 got value 0.04025579984426745
For 8 got value -1.5664730318699973
For 9 got value -0.8474847232150614


np.uint8(3)

In [23]:
y_test[30]

np.uint8(3)

In [24]:
baseperm = np.arange(10)

In [25]:
perms = []
left1 = [np.append(0,i) for i in itertools.combinations(np.arange(1,10),4)]
right1 = [np.setdiff1d(baseperm,left) for left in left1]
left2 = [np.array(i) for i in itertools.combinations(np.arange(5),3)]
right2 = [np.setdiff1d(np.arange(5),left) for left in left2]
left3 = [np.array(i) for i in itertools.combinations(np.arange(3),2)]
right3 = [np.setdiff1d(np.arange(3),left) for left in left3]
for i1 in range(len(left1)):
    for i2 in range(len(left2)):
        for i3 in range(len(left2)):
            for i4 in range(len(left3)):
                for i5 in range(len(left3)):
                    # Derefrence lists
                    l1 = left1[i1]
                    r1 = right1[i1]
                    l2 = left2[i2]
                    r2 = right2[i2]
                    l3 = left2[i3]
                    r3 = right2[i3]
                    l4 = left3[i4]
                    r4 = right3[i4]
                    l5 = left3[i5]
                    r5 = right3[i5]
                    
                    ll = l1[l2]
                    lr = l1[r2]
                    rl = r1[l3]
                    rr = r1[r3]

                    entry = np.concat([ll[l4],ll[r4],lr,rl[l5],rl[r5],rr])
                    perms.append(entry)

In [26]:
perms[0]

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [27]:
SeperatingSets = [[[0,1,2,3,4],[5,6,7,8,9]],
                  [[0,1,2],[3,4]],
                  [[5,6,7],[8,9]],
                  [[0,1],[2]],
                  [[3],[4]],
                  [[5,6],[7]],
                  [[8],[9]],
                  [[0],[1]],
                  [[5],[6]]
                 ]

In [28]:
ProgressIndexes = [
    [[False,1],[False,2]],
    [[False,3],[False,4]],
    [[False,5],[False,6]],
    [[False,7],[True,2]],
    [[True,3],[True,4]],
    [[False,8],[True,7]],
    [[True,8],[True,9]],
    [[True,0],[True,1]],
    [[True,5],[True,6]]
]

In [29]:
class seperator_data(NamedTuple):
    perm: tuple
    ws: np.array
    cs: np.array

In [47]:
# Binary seperator
def gen_perm_sep(perm):
    ws = []
    cs = []
    for i in range(9):
        S0 = SeperatingSets[i][0]
        S1 = SeperatingSets[i][1]
        vals0 = baseperm[np.isin(perm,S0)]
        vals1 = baseperm[np.isin(perm,S1)]
        data0 = x_train[np.isin(y_train,vals0)]
        data1 = x_train[np.isin(y_train,vals1)]
        cov0 = np.cov(data0,rowvar=False)
        cov1 = np.cov(data1,rowvar=False)
        mean0 = np.mean(data0,axis=0)
        mean1 = np.mean(data1,axis=0)
        wt = np.linalg.pinv(cov0+cov1,hermitian=True)@(mean0-mean1)
        wp = wt/np.linalg.norm(wt)
        vals0 = np.sort(wp@np.transpose(data0))
        vals1 = np.sort(wp@np.transpose(data1))
        tosearch = vals0[vals0< vals1[-1]]
        best = np.inf
        for i in range(len(tosearch)):
            score = np.sum(vals1 >tosearch[i])+i
            if score <best:
                best = score
                ct = tosearch[i]
        ws.append(wp)
        cs.append(ct)
    sep_data = seperator_data(perm,ws,cs)
    return sep_data

In [ ]:
sep_data = gen_perm_sep(perms[10])

In [32]:
def classify_bin(datapoint,sep_data,verbose = False):
    perm = sep_data.perm
    ws = sep_data.ws
    cs = sep_data.cs
    index = 0
    while index <10:
        w = ws[index]
        c = cs[index]
        ind2 = 0 if ((w@datapoint)>=c) else 1
        nextval = ProgressIndexes[index][ind2]
        if verbose:
            print(nextval)
        if nextval[0]:
            return baseperm[np.equal(perm,nextval[1])][0]
        else:
            index = nextval[1]

In [33]:
sep_data.cs

[np.float64(-1.0890560807845828),
 np.float64(-0.6634184170744686),
 np.float64(-0.975006721617035),
 np.float64(-1.0763382206250673),
 np.float64(-0.006547765794970477),
 np.float64(-0.4473533544447931),
 np.float64(0.02176216634523282),
 np.float64(-0.4086196292515949),
 np.float64(-0.016763360187110648)]

In [34]:
classify_bin(x_test[28],sep_data)

np.int64(0)

In [35]:
y_test[28]

np.uint8(0)

In [36]:
np.sum([classify_bin(x_train[i],sep_data) for i in range(60000)] != y_train)

np.int64(11169)

In [53]:
# Find the best permutation
bestval = np.inf
def process_perms(perm):
    #perm = perms[j]
    sep_data_t = gen_perm_sep(perm)
    perf = np.sum([classify_bin(x_train[i],sep_data_t) for i in range(60000)] != y_train)
    return perf
r = [process_perms(perm) for perm in tqdm(perms)]
#process_perms(1)
#r = process_map(process_perms,perms,chunksize=2000)
#with Pool(1) as pool:
#    r = list(pool.map(process_perms,perms))

  0%|          | 0/113400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
(sep_data.ws[0]@np.transpose(x_train))[0]

In [ ]:
x_train